# 02 - Preprocessing

Στόχος αυτού του notebook:
1. Φόρτωση όλων των tables
2. Date parsing και cleaning
3. Joins για τη δημιουργία ενός ενιαίου dataset
4. Handling missing values (κυρίως oil)
5. Holiday handling με conditional logic
6. Αποθήκευση του processed dataset για χρήση στα επόμενα notebooks

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print("Setup OK")

Setup OK


In [3]:
train = pd.read_csv('../data/raw/train.csv')
stores = pd.read_csv('../data/raw/stores.csv')
oil = pd.read_csv('../data/raw/oil.csv')
holidays = pd.read_csv('../data/raw/holidays_events.csv')
transactions = pd.read_csv('../data/raw/transactions.csv')

print(f"train: {train.shape}")
print(f"stores: {stores.shape}")
print(f"oil: {oil.shape}")
print(f"holidays: {holidays.shape}")
print(f"transactions: {transactions.shape}")

train: (3000888, 6)
stores: (54, 5)
oil: (1218, 2)
holidays: (350, 6)
transactions: (83488, 3)


In [4]:
# Μετατροπή σε datetime σε όλους τους πίνακες
train['date'] = pd.to_datetime(train['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])
transactions['date'] = pd.to_datetime(transactions['date'])

# Επιβεβαίωση
print("Datetime conversion done:")
print(f"  train.date: {train['date'].dtype}")
print(f"  oil.date: {oil['date'].dtype}")
print(f"  holidays.date: {holidays['date'].dtype}")
print(f"  transactions.date: {transactions['date'].dtype}")

Datetime conversion done:
  train.date: datetime64[us]
  oil.date: datetime64[us]
  holidays.date: datetime64[us]
  transactions.date: datetime64[us]


## Oil price preprocessing

Το oil dataset έχει:
- 43 missing values (NaN) σε Σαββατοκύριακα/αργίες
- Ολόκληρες ημερομηνίες που λείπουν (π.χ. 2013-01-05/06)

Στρατηγική:
1. Φτιάχνουμε πλήρες χρονικό index από την πρώτη μέχρι την τελευταία ημερομηνία του train
2. Reindex το oil για να καλύψει όλες τις ημερομηνίες
3. Forward fill για να γεμίσουμε τα κενά (standard τεχνική για financial data)
4. Backward fill για την πρώτη γραμμή (αν είναι NaN)

In [5]:
# Φτιάχνουμε πλήρες ημερολόγιο
date_range = pd.date_range(start=train['date'].min(), end=train['date'].max(), freq='D')
print(f"Πλήρες range: {date_range.min()} έως {date_range.max()}")
print(f"Σύνολο ημερών: {len(date_range)}")

# Reindex του oil για να καλύπτει όλες τις ημερομηνίες
oil_full = oil.set_index('date').reindex(date_range).reset_index()
oil_full.columns = ['date', 'oil_price']

# Πόσα NaN τώρα;
print(f"\nNaN πριν το fill: {oil_full['oil_price'].isnull().sum()}")

# Forward fill, μετά backward fill για τυχόν NaN στην αρχή
oil_full['oil_price'] = oil_full['oil_price'].ffill().bfill()

print(f"NaN μετά το fill: {oil_full['oil_price'].isnull().sum()}")
print(f"\nΠρώτες γραμμές:")
oil_full.head(10)

Πλήρες range: 2013-01-01 00:00:00 έως 2017-08-15 00:00:00
Σύνολο ημερών: 1688

NaN πριν το fill: 525
NaN μετά το fill: 0

Πρώτες γραμμές:


,date,oil_price
0,2013-01-01,93.14
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-05,93.12
5,2013-01-06,93.12
6,2013-01-07,93.20
7,2013-01-08,93.21
8,2013-01-09,93.08
9,2013-01-10,93.81


## Joins

Ενώνουμε όλους τους πίνακες σε ένα ενιαίο dataset (`df`).
Ξεκινάμε με τους πιο απλούς joins:
1. `stores` — προσθέτει city, state, type, cluster (left join on store_nbr)
2. `oil_full` — προσθέτει oil_price (left join on date)
3. `transactions` — προσθέτει transactions (left join on store_nbr + date)

Το `holidays` θα το χειριστούμε ξεχωριστά γιατί χρειάζεται conditional logic.

In [6]:
# Ξεκινάμε με το train ως βάση
df = train.copy()
print(f"Αρχικό shape: {df.shape}")

# Join 1: stores
df = df.merge(stores, on='store_nbr', how='left')
print(f"Μετά το stores join: {df.shape}")
print(f"Νέες στήλες: {[c for c in df.columns if c not in train.columns]}")

Αρχικό shape: (3000888, 6)
Μετά το stores join: (3000888, 10)
Νέες στήλες: ['city', 'state', 'type', 'cluster']


In [7]:
# Join 2: oil
df = df.merge(oil_full, on='date', how='left')
print(f"Μετά το oil join: {df.shape}")

# Επιβεβαίωση: δεν πρέπει να υπάρχουν NaN στο oil_price
print(f"NaN στο oil_price: {df['oil_price'].isnull().sum()}")

Μετά το oil join: (3000888, 11)
NaN στο oil_price: 0


In [8]:

# Join 3: transactions
df = df.merge(transactions, on=['store_nbr', 'date'], how='left')
print(f"Μετά το transactions join: {df.shape}")

# Πόσα NaN έχει το transactions;
print(f"NaN στο transactions: {df['transactions'].isnull().sum()}")
print(f"Ποσοστό: {100 * df['transactions'].isnull().sum() / len(df):.1f}%")

Μετά το transactions join: (3000888, 12)
NaN στο transactions: 245784
Ποσοστό: 8.2%


In [9]:
print(f"Τελικό shape: {df.shape}")
print(f"\nΣτήλες: {list(df.columns)}")
print(f"\nDtypes:")
print(df.dtypes)
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nΔείγμα 5 γραμμών:")
df.head()

Τελικό shape: (3000888, 12)

Στήλες: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'oil_price', 'transactions']

Dtypes:
id                       int64
date            datetime64[us]
store_nbr                int64
family                     str
sales                  float64
onpromotion              int64
city                       str
state                      str
type                       str
cluster                  int64
oil_price              float64
transactions           float64
dtype: object

Missing values per column:
id                   0
date                 0
store_nbr            0
family               0
sales                0
onpromotion          0
city                 0
state                0
type                 0
cluster              0
oil_price            0
transactions    245784
dtype: int64

Δείγμα 5 γραμμών:


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,oil_price,transactions
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,93.14,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,93.14,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,93.14,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,93.14,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,93.14,NaN


In [10]:
# Πάρε ένα δείγμα γραμμών με transactions=NaN
nan_transactions = df[df['transactions'].isnull()]
print(f"Γραμμές με NaN transactions: {len(nan_transactions):,}")
print(f"\nΑπό αυτές, πόσες έχουν sales=0;")
print(f"  sales=0: {(nan_transactions['sales'] == 0).sum():,} ({100*(nan_transactions['sales'] == 0).sum()/len(nan_transactions):.1f}%)")
print(f"  sales>0: {(nan_transactions['sales'] > 0).sum():,}")

# Δείγμα γραμμών όπου transactions=NaN αλλά sales>0 (αν υπάρχουν)
weird_rows = nan_transactions[nan_transactions['sales'] > 0]
if len(weird_rows) > 0:
    print(f"\nΠαράξενες γραμμές (NaN transactions αλλά πούλησαν):")
    print(weird_rows.head(10))
else:
    print("\nΌλα τα NaN transactions αντιστοιχούν σε sales=0 ✓")

Γραμμές με NaN transactions: 245,784

Από αυτές, πόσες έχουν sales=0;
  sales=0: 242,536 (98.7%)
  sales>0: 3,248

Παράξενες γραμμές (NaN transactions αλλά πούλησαν):
            id       date  store_nbr        family   sales  onpromotion  \
301191  301191 2013-06-19         10    AUTOMOTIVE     3.0            0   
301194  301194 2013-06-19         10     BEVERAGES   515.0            0   
301196  301196 2013-06-19         10  BREAD/BAKERY    94.0            0   
301198  301198 2013-06-19         10      CLEANING   695.0            0   
301199  301199 2013-06-19         10         DAIRY    43.0            0   
301200  301200 2013-06-19         10          DELI   186.0            0   
301201  301201 2013-06-19         10          EGGS    28.0            0   
301202  301202 2013-06-19         10  FROZEN FOODS    12.0            0   
301203  301203 2013-06-19         10     GROCERY I  1785.0            0   
301204  301204 2013-06-19         10    GROCERY II    10.0            0   

       

In [11]:
# Πριν το fill - πόσα NaN
print(f"NaN πριν: {df['transactions'].isnull().sum():,}")

# Stage 1: όπου sales=0, transactions=0 (κλειστά καταστήματα)
mask_closed = (df['transactions'].isnull()) & (df['sales'] == 0)
df.loc[mask_closed, 'transactions'] = 0
print(f"Μετά το stage 1 (sales=0 → trans=0): {df['transactions'].isnull().sum():,}")

# Stage 2: όπου sales>0 αλλά transactions=NaN
# Imputation με τον μέσο όρο του καταστήματος (μόνο από non-zero μέρες)
store_avg_transactions = df[df['transactions'] > 0].groupby('store_nbr')['transactions'].mean()
print(f"\nΜέσος όρος transactions ανά κατάστημα (πρώτα 10):")
print(store_avg_transactions.head(10).round(0))

# Εφαρμογή του imputation
def fill_transactions(row):
    if pd.isnull(row['transactions']):
        return store_avg_transactions.get(row['store_nbr'], 0)
    return row['transactions']

df['transactions'] = df.apply(fill_transactions, axis=1)
print(f"\nNaN μετά το stage 2: {df['transactions'].isnull().sum():,}")

NaN πριν: 245,784
Μετά το stage 1 (sales=0 → trans=0): 3,248

Μέσος όρος transactions ανά κατάστημα (πρώτα 10):
store_nbr
1     1524.0
2     1920.0
3     3202.0
4     1503.0
5     1400.0
6     1829.0
7     1789.0
8     2767.0
9     2098.0
10     987.0
Name: transactions, dtype: float64

NaN μετά το stage 2: 0


In [12]:
print(f"Συνολικά NaN στο dataset:")
print(df.isnull().sum())
print(f"\nStatistics για transactions:")
print(df['transactions'].describe().round(2))

Συνολικά NaN στο dataset:
id              0
date            0
store_nbr       0
family          0
sales           0
onpromotion     0
city            0
state           0
type            0
cluster         0
oil_price       0
transactions    0
dtype: int64

Statistics για transactions:
count    3000888.00
mean        1557.65
std         1032.51
min            0.00
25%          932.00
50%         1332.00
75%         1978.00
max         8359.00
Name: transactions, dtype: float64


## Holiday handling

Σύνθετο join γιατί:
- National holidays αφορούν όλα τα stores
- Regional/Local αφορούν συγκεκριμένη επαρχία/πόλη
- Transferred=True σημαίνει "δεν είναι πραγματική αργία αυτή τη μέρα"

Δημιουργούμε 3 ξεχωριστά flags που θα γίνουν OR στο τέλος.

In [13]:
# Φιλτράρισμα: αφαιρούμε τις transferred αργίες
# (δεν είναι πραγματικές αργίες αυτή τη μέρα)
holidays_clean = holidays[holidays['transferred'] == False].copy()
print(f"Αρχικά: {len(holidays)} γραμμές")
print(f"Μετά την αφαίρεση transferred: {len(holidays_clean)} γραμμές")

# Επίσης φιλτράρουμε τα Work Day (αντίστροφο της αργίας - μέρα εργασίας)
holidays_clean = holidays_clean[holidays_clean['type'] != 'Work Day']
print(f"Μετά την αφαίρεση Work Day: {len(holidays_clean)} γραμμές")

# Δες τι έχουμε ανά locale
print(f"\nΑνά locale:")
print(holidays_clean['locale'].value_counts())

Αρχικά: 350 γραμμές
Μετά την αφαίρεση transferred: 338 γραμμές
Μετά την αφαίρεση Work Day: 333 γραμμές

Ανά locale:
locale
National    161
Local       148
Regional     24
Name: count, dtype: int64


In [14]:
# Χωρισμός holidays σε National, Regional, Local
national_holidays = holidays_clean[holidays_clean['locale'] == 'National']['date'].unique()
regional_holidays = holidays_clean[holidays_clean['locale'] == 'Regional'][['date', 'locale_name']].copy()
local_holidays = holidays_clean[holidays_clean['locale'] == 'Local'][['date', 'locale_name']].copy()

print(f"National holidays: {len(national_holidays)} μοναδικές ημερομηνίες")
print(f"Regional holidays: {len(regional_holidays)} γραμμές")
print(f"Local holidays: {len(local_holidays)} γραμμές")

National holidays: 155 μοναδικές ημερομηνίες
Regional holidays: 24 γραμμές
Local holidays: 148 γραμμές


In [15]:
# Flag 1: National holiday
df['is_national_holiday'] = df['date'].isin(national_holidays)

# Flag 2: Regional holiday (χρειάζεται match σε date + state)
# Φτιάχνουμε ένα set από (date, state) tuples
regional_set = set(zip(regional_holidays['date'], regional_holidays['locale_name']))
df['is_regional_holiday'] = list(zip(df['date'], df['state']))
df['is_regional_holiday'] = df['is_regional_holiday'].apply(lambda x: x in regional_set)

# Flag 3: Local holiday (date + city)
local_set = set(zip(local_holidays['date'], local_holidays['locale_name']))
df['is_local_holiday'] = list(zip(df['date'], df['city']))
df['is_local_holiday'] = df['is_local_holiday'].apply(lambda x: x in local_set)

# Συνολικό flag
df['is_holiday'] = df['is_national_holiday'] | df['is_regional_holiday'] | df['is_local_holiday']

print("Holiday flags created ✓")
print(f"\nΠοσοστά:")
print(f"  National: {100 * df['is_national_holiday'].mean():.1f}%")
print(f"  Regional: {100 * df['is_regional_holiday'].mean():.1f}%")
print(f"  Local: {100 * df['is_local_holiday'].mean():.1f}%")
print(f"  Any:      {100 * df['is_holiday'].mean():.1f}%")

Holiday flags created ✓

Ποσοστά:
  National: 7.8%
  Regional: 0.0%
  Local: 0.4%
  Any:      8.2%


In [16]:
# Έλεγχος: σε ημέρες με holiday, είναι οι πωλήσεις διαφορετικές;
print("Μέσες πωλήσεις (μόνο non-zero) ανά holiday status:")
print(df[df['sales'] > 0].groupby('is_holiday')['sales'].agg(['mean', 'median', 'count']).round(2))

# Δείγμα γραμμών με τα νέα flags
print(f"\nΔείγμα με τα νέα flags:")
df[df['is_holiday']].sample(5, random_state=42)[['date', 'store_nbr', 'city', 'state', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'sales']]

Μέσες πωλήσεις (μόνο non-zero) ανά holiday status:
              mean  median    count
is_holiday                         
False       513.57    78.0  1888954
True        599.15    84.0   172804

Δείγμα με τα νέα flags:


,date,store_nbr,city,state,is_national_holiday,is_regional_holiday,is_local_holiday,sales
1552534,2015-05-24,20,Quito,Pichincha,True,False,False,31.000
1151973,2014-10-10,31,Babahoyo,Los Rios,True,False,False,212.687
962121,2014-06-25,54,El Carmen,Manabi,True,False,False,0.000
75578,2013-02-12,3,Quito,Pichincha,True,False,False,1093.000
649894,2014-01-01,43,Esmeraldas,Esmeraldas,True,False,False,0.000
